# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [6]:
%pip -q install duckdb huggingface_hub


In [7]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [8]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [9]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [10]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435,8.0,0.136531,0.239391,1223.0,1353.0,0.903917
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875,38.0,0.146092,0.336034,900.0,2173.0,0.414174
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556,42.0,0.064587,0.803004,103.0,1146.0,0.089878
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698,4.0,0.384248,0.408115,38.0,87.0,0.436782
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371,3.0,0.104607,0.860845,15.0,36.0,0.416667


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [11]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.538     0.334     0.412      9389
           1      0.683     0.833     0.751     16162

    accuracy                          0.650     25551
   macro avg      0.610     0.584     0.581     25551
weighted avg      0.629     0.650     0.626     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


## Per-client validation

**Question.** Does the decline model generalize to clients it has never seen?

**Why this matters.** The first model used a random split, so pages from the same
client appeared in both training and test sets. A model can score well there simply
by recognising client-specific habits (for example, a client whose pages all
fluctuate in similar ways), not by learning anything transferable.

**Method.** I split by `client_hash_id` using `GroupShuffleSplit`, so each client's
pages fall entirely in training or entirely in test. The model, features and label
are unchanged: a random forest on `imp_prev30`, `visible_queries`, `rare_share`,
`anon_share` and `top_query_share`, predicting `is_declining`
(last-30-day impressions below 80% of the previous 30 days).

**Reading the result.** I compare the test accuracy against the base rate of
always predicting the majority class. If the score falls toward or below the base
rate, the earlier result was partly an artefact of the random split. If it holds,
the signal is more likely to transfer across clients.

**Caveat.** This is observed, directional evidence for decision support. It does
not show that any feature causes a decline.

In [12]:
from sklearn.model_selection import GroupShuffleSplit

# Group pages by client so no client appears in both train and test.
groups = model_data['client_hash_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))

# Build train and test sets from those row positions.
X_tr2, X_te2 = X.iloc[tr_idx], X.iloc[te_idx]
y_tr2, y_te2 = y.iloc[tr_idx], y.iloc[te_idx]

# Sanity check: how many clients ended up on each side?
print('clients in train:', groups.iloc[tr_idx].nunique())
print('clients in test: ', groups.iloc[te_idx].nunique())

# Same model as before, so any change in score comes from the split alone.
model2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)

# Compare against the "always guess the majority class" baseline.
print(f'base rate (always predict majority): {max(y_te2.mean(), 1 - y_te2.mean()):.3f}')
print(classification_report(y_te2, model2.predict(X_te2), digits=3))

clients in train: 36
clients in test:  13
base rate (always predict majority): 0.677
              precision    recall  f1-score   support

           0      0.414     0.465     0.438     16706
           1      0.728     0.686     0.706     34996

    accuracy                          0.614     51702
   macro avg      0.571     0.575     0.572     51702
weighted avg      0.627     0.614     0.619     51702



**Result.** With a per-client split (36 clients train, 13 test; 51,702 test pages),
accuracy is 0.614 against a majority-class base rate of 0.677. The model does not beat
the baseline on accuracy. The earlier random-split score (0.650 vs base rate 0.633)
was therefore partly inflated by client-specific patterns shared across train and test.
Precision for "declining" is 0.728 against a 0.677 prevalence, and for "not declining"
0.414 against 0.323, so there is a small amount of signal, but accuracy alone does not
show it.

**Interpretation.** These features carry weak, directional evidence about which pages
decline, and they do not generalize strongly across clients. Next I check ROC-AUC
(which does not depend on a decision threshold) and repeat the split with several
random seeds to see how stable this is.

In [14]:
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

def make_model(seed):
    # Smaller forest so the comparison runs quickly.
    return RandomForestClassifier(n_estimators=60, max_depth=12, min_samples_leaf=50,
                                  random_state=seed, n_jobs=-1)

rows = []
for seed in range(3):
    # Per-client split: test clients are never seen in training.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr, te = next(gss.split(X, y, groups))
    m = make_model(seed).fit(X.iloc[tr], y.iloc[tr])
    rows.append(('per-client', seed, roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1])))

    # Random split: pages from the same client can be in both sets.
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = make_model(seed).fit(Xa, ya)
    rows.append(('random', seed, roc_auc_score(yb, m.predict_proba(Xb)[:, 1])))

res = pd.DataFrame(rows, columns=['split', 'seed', 'roc_auc'])
print(res.round(3))
print(res.groupby('split')['roc_auc'].agg(['mean', 'std']).round(3))

        split  seed  roc_auc
0  per-client     0    0.618
1      random     0    0.657
2  per-client     1    0.670
3      random     1    0.655
4  per-client     2    0.616
5      random     2    0.662
             mean    std
split                   
per-client  0.635  0.031
random      0.658  0.003


**Result: stability across splits.** Over three seeds with small random forests, ROC-AUC is
0.658 (std 0.003) under a random split and 0.635 (std 0.031) under a per-client split.
Both are above 0.5, so the five features carry a weak, directional signal about which
pages lose impressions. The per-client score is slightly lower and much more variable
(0.616 to 0.670 across seeds), which suggests that how well the model transfers depends
on which clients are held out. With only 49 clients and three seeds, the evidence does not
support a claim that the model fails to generalize, and it does not support a claim that
it generalizes well. It supports decision-support use only: ranking pages for human review,
not automatic action.

In [15]:
sets = {
    'imp_prev30 only': ['imp_prev30'],
    'query features only': ['visible_queries', 'rare_share', 'anon_share', 'top_query_share'],
    'all five': feature_cols,
}

rows = []
for name, cols in sets.items():
    for seed in range(3):
        gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
        tr, te = next(gss.split(X, y, groups))
        m = make_model(seed).fit(X.iloc[tr][cols], y.iloc[tr])
        auc = roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te][cols])[:, 1])
        rows.append((name, seed, auc))

abl = pd.DataFrame(rows, columns=['features', 'seed', 'roc_auc'])
print(abl.groupby('features')['roc_auc'].agg(['mean', 'std']).round(3))

                      mean    std
features                         
all five             0.635  0.031
imp_prev30 only      0.496  0.026
query features only  0.589  0.021


**Result: which features carry the signal.** Under per-client splits (3 seeds), ROC-AUC
is 0.496 for `imp_prev30` alone, 0.589 for the four query features alone, and 0.635 for
all five. Prior impression volume alone is no better than chance, even though it is part of
the label definition, so the model is not simply recovering the label's arithmetic. The query
features carry most of the signal, and adding impression volume improves them, which suggests
the two interact. Differences under about 0.03 are within seed-to-seed noise.

**Limitation.** The query features are aggregated over a 90-day window that overlaps the
30-day window used to define the label. They are therefore contemporaneous correlates of
decline, not forecasts made before it happened. Results are observed and directional, for
decision support only.